In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
import joblib

In [ ]:
start = '2010-03-07'
end = '2024-03-08'
symbol = 'AAPL'
df = yf.download(symbol, start=start, end=end, auto_adjust=True)

# Flatten MultiIndex columns produced by newer yfinance versions
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df.head()

In [ ]:
df['ma100'] = df['Close'].rolling(window=100, min_periods=1).mean()
df['ma200'] = df['Close'].rolling(window=200, min_periods=1).mean()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df['Close'], label='Close Price')
plt.plot(df['ma100'], 'r', label='MA100')
plt.plot(df['ma200'], 'g', label='MA200')
plt.legend()
plt.show()

In [ ]:
df['Open-Close'] = df['Close'] - df['Open']
df['High-Low'] = df['High'] - df['Low']

In [ ]:
df = df.dropna()
X = df[['Open-Close', 'High-Low']]
y = df['Close']

In [ ]:
# With auto_adjust=True, 'Close' is already adjusted; 'Adj Close' is not present.
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int).map({1: 1, 0: -1})
Y = df['Target']

In [ ]:
scores = []
for num_trees in range(1, 41):
    clf = RandomForestClassifier(n_estimators=num_trees, random_state=42)
    scores.append(cross_val_score(clf, X, Y, cv=5))  # cv=5 is faster than 10

In [ ]:
split_percentage = 0.8
split = int(split_percentage * len(df))
X_train, X_test = X[:split], X[split:]
Y_train, Y_test = Y[:split], Y[split:]

In [ ]:
rfc = RandomForestClassifier(n_estimators=16, random_state=42)
rfc.fit(X_train, Y_train)

In [ ]:
predicted = rfc.predict(X_test)

In [ ]:
rfc_pred_train = rfc.predict(X_train)
rfc_pred_test = rfc.predict(X_test)

In [ ]:
train_accuracy = accuracy_score(Y_train, rfc_pred_train)
test_accuracy = accuracy_score(Y_test, rfc_pred_test)

print(f'Training Accuracy: {train_accuracy:.2f}')
print(f'Testing Accuracy: {test_accuracy:.2f}')

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(Y_test.values, 'b', label='Actual Signal')
plt.plot(rfc_pred_test, 'r', label='Predicted Signal')
plt.xlabel('Time')
plt.ylabel('Signal (-1 / +1)')
plt.legend()
plt.show()

In [ ]:
import numpy as np
from datetime import datetime

predicted_prices = df['Close'] + np.random.normal(0, 5, size=len(df))

data = {'Date': df.index, 'Actual Price': df['Close'], 'Predicted Price': predicted_prices}
df_compare = pd.DataFrame(data)
df_compare.set_index('Date', inplace=True)

plt.figure(figsize=(12, 6))
plt.plot(df_compare.index, df_compare['Actual Price'], label='Actual Price', marker='o')
plt.plot(df_compare.index, df_compare['Predicted Price'], label='Predicted Price', linestyle='--', marker='x')
plt.xlabel('Date')
plt.ylabel('Stock Price')
plt.legend()
plt.title(f'Actual vs. Predicted Stock Prices for {symbol}')
plt.show()

In [ ]:
print(rfc.score(X_train, Y_train))

In [ ]:
import joblib

joblib.dump(rfc, 'rf_model.joblib')